In [ ]:
import osmnx as ox

G = ox.load_graphml("budapest.graphml")

117248 218595
{'y': 47.5135409, 'x': 19.0580139, 'street_count': 3}
Hello


In [ ]:
'''
Ez egy próbacella, tele mindenféle ötletelgetéssel és megjegyzésekkel magamnak asfjhalkhflakjhsdlkajlkasjdlk...

#Így lehet G csúcshalmazára, élhalmazára és egy adott csúcsára hivatkozni
print(len(G.nodes), len(G.edges))
node_id = list(G.nodes)[0]
print(G.nodes[node_id])


place = "deak ter, Budapest, Hungary"

#Ez az ox.goecode() megtalálja a koordinátákat, és fuzzy match-et is végez
#De: internet kell hozzá, és viszonylag lassú
point = ox.geocode(place)

print(point)  # (lat, lon)
node = ox.distance.nearest_nodes(G, point[1], point[0])

#Output: (47.4983508, 19.0535905)
'''

(47.4983508, 19.0535905)


In [ ]:
import json
import osmnx as ox
from rapidfuzz import process

CACHE_FILE = "place_coords.json"

#Meglévő dictionary betöltése, vagy ha nincs, új csinálása
try:
    with open(CACHE_FILE, "r", encoding="utf-8") as f:
        place_coords = json.load(f)
except FileNotFoundError:
    place_coords = {}
    print("Üres", CACHE_FILE, "könyvtár létrehozva")
    with open(CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(place_coords, f, ensure_ascii=False, indent=2)


def save_cache():
    with open(CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(place_coords, f, ensure_ascii=False, indent=2)


def find_coords(name, G):
    #Először megnézzük, hogy van-e a place_coords-ban már ilyen név
    if place_coords:
        best_match = process.extractOne(name, place_coords.keys())
        if best_match and best_match[1] >= 70:
            best_name = best_match[0]
            lat, lon = place_coords[best_name]
            node = ox.distance.nearest_nodes(G, lon, lat) #az ox-nak (lon, lat) sorrendben kell megadni a koordinátákat
            return best_name, node, lat, lon
    #Ha nincs, akkor a geocode-ból kell lekérdeznünk (kell internet + lassú)
    search_term = name + ", Budapest, Hungary"
    lat, lon = ox.geocode(search_term)
    node = ox.distance.nearest_nodes(G, lon, lat) #az ox-nak (lon, lat) sorrendben kell megadni a koordinátákat
    #Ebben az ágban el is kell mentenünk a frissített könyvtárat
    place_coords[name] = [lat, lon]
    save_cache()

    return name, node, lat, lon


#példa hívás:
find_coords("Deák Ferenc tér", G)

('Deák Ferenc tér', 11365115119, 47.4983508, 19.0535905)